In [1]:
from sequana import DNA
from sequana import FastA
import pandas as pd
import numpy as np
from sklearn.preprocessing import  StandardScaler
import re
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
from scipy.signal import argrelextrema
from itertools import combinations
from scipy.signal import peak_widths



In [2]:
def count_homopolymers(seq, min_length=5):
    # Ex : trouve AAAAA ou TTTTT, etc.
    pattern = re.compile(rf"(A{{{min_length},}}|T{{{min_length},}}|C{{{min_length},}}|G{{{min_length},}})")
    return len(pattern.findall(seq.upper()))

def load_fasta(fasta_path, window_size=100):
    f = FastA(fasta_path)
    data = []


    for maseq in f:

        print(maseq.name)
        
        features = []

        s = DNA(maseq.sequence.upper())
        seq = maseq.sequence.upper()
        s.window = window_size

       

        #Homopolymere
        X2 = []
        X3 = []

        
        for i in range(0, len(seq)-2, 1):
            
            window = seq[max(0, i - window_size//2):min(i+window_size//2,len(seq))]
            nb = count_homopolymers(window, min_length=2)
            X2.append(nb)

            nb = count_homopolymers(window, min_length=3)
            X3.append(nb)


            
        X2 = X2[5000:-5000]
        X3 = X3[5000:-5000]



        df= pd.DataFrame({
            'X2':X2,
            'X3':X3
        })

        
        data.append(df)

            
    return data

 
#dataFriedlin2021 = load_fasta("../data/Fasta/Major/TriTrypDB-68_LmajorFriedlin2021_Genome.fasta",200)
#dataLV39c5 = load_fasta("../data/Fasta/Major/TriTrypDB-68_LmajorLV39c5_Genome.fasta",200)
#dataSD75 = load_fasta("../data/Fasta/Major/TriTrypDB-68_LmajorSD75.1_Genome.fasta",200)
dataBest = load_fasta("../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta",200)



LmjF.01
LmjF.02
LmjF.03
LmjF.04
LmjF.05
LmjF.06
LmjF.07
LmjF.08
LmjF.09
LmjF.10
LmjF.11
LmjF.12
LmjF.13
LmjF.14
LmjF.15
LmjF.16
LmjF.17
LmjF.18
LmjF.19
LmjF.20
LmjF.21
LmjF.22
LmjF.23
LmjF.24
LmjF.25
LmjF.26
LmjF.27
LmjF.28
LmjF.29
LmjF.30
LmjF.31
LmjF.32
LmjF.33
LmjF.34
LmjF.35
LmjF.36


In [5]:

vecteur_Debut = []
vecteur_Fin = []

vecteur_longeur = []

df = pd.read_csv("../data/Centromere_Positions/pos_libre.csv")
pos_libre = {
    int(row.Chromosome): (int(row.Start), int(row.End))
    for row in df.itertuples(index=False)
}

g = 0
#data = dataLV39c5
#data = dataSD75 #Chromosome a redefnir
#data = dataFriedlin2021
data = dataBest

for i in range(0,36):
    temp_ =  data[i]['X3']*data[i]['X2']

    telo = 40000
    temp_ = temp_[telo:]

    mean = np.mean(temp_)

    peaks, properties = find_peaks(temp_,  prominence=mean*2, distance=20)

    window = 3000
    peak_medians = []

 



        #Moyenne
    peak_means = []
    for peak in peaks:
        start = max(0, peak - window//2)
        end = min(len(temp_), peak + window//2)
        mean_val = np.mean(temp_[start:end])
        peak_medians.append((peak, mean_val))
    
  #  for peak in peaks:
  #      start = max(0, peak - window//2)
  #      end = min(len(temp_), peak + window//2 + 1)
    
   #     neighborhood = temp_[start:end]
   #     if len(neighborhood) > 0:
   #         median_val = np.median(neighborhood)
   #         peak_medians.append((peak, median_val))
    
    # Trouver le pic avec la médiane la plus élevée
    if peak_medians:
        peak_max, max_median = max(peak_medians, key=lambda x: x[1])
        max_index = peak_max


        #Chercher la taille du centromere


        # Appliquer un filtre pour lisser
        temp_X2_3 = uniform_filter1d(temp_, size=1500)
        temp_X2_3 = temp_X2_3[max_index-25000:max_index+25000]
        

        
        # Détection des pics
        peaks, properties = find_peaks(temp_X2_3, prominence=5, distance=200)

            
        # Trouver le pic avec la plus grande **prominence**
        if len(peaks) > 0:
            prominences = properties['prominences']
            peak_index = np.argmax(prominences)
            peak_max = peaks[peak_index]
        
            # Calcule la largeur du pic à mi-hauteur
            widths_result = peak_widths(temp_X2_3, peaks, rel_height=0.4)
        
            taille = widths_result[0][peak_index]
            seuil = widths_result[1][peak_index]
            debut = widths_result[2][peak_index]
            fin = widths_result[3][peak_index]


            debut = int(debut)
            fin = int(fin)
            
            marge = 750
            finish = True
            distance = []
            temp_fin = []
            while finish:
                
                recherche = False
                limiteFin = min(len(temp_X2_3), fin + marge)
                pos = fin

                while pos < limiteFin and recherche == False:
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos += 1
                if recherche == True:
                     temp_fin.append(fin)
                     distance.append(0)
                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos += 1
                        distance[len(distance)-1] += 1

                     fin = pos
                else: 
                    finish = False

            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                fin = temp_fin[t]
                t = t -1 
            #Avant
            finish = True
            distance = 0
            distance = []
            temp_debut = []

            while finish:
                recherche = False
                limiteDebut = max(0, debut - marge)
                pos = debut
                while pos > limiteDebut and recherche == False:
 
                    if  temp_X2_3[pos] > seuil:
                        recherche = True
                    pos -= 1
        
                if recherche == True:
                     temp_debut.append(debut)
                     distance.append(0)

                     while pos + 1 < len(temp_X2_3) and temp_X2_3[pos] > seuil:
                        pos -= 1
                        distance[len(distance)-1] += 1
                     debut = pos
                else:
                    finish = False


            t = len(distance)-1
            while t >= 0 and distance[t] < 200:
                debut = temp_debut[t]
                t = t -1 
                

            taille = fin - debut


            


            
            debut = debut+max_index-25000+5000+telo
            fin = fin+max_index-25000+5000+telo
            debut = int(debut)
            fin = int(fin)
            max_index = max_index+5000+telo
            # Affichage

            pos_start, pos_end = pos_libre[i+1]

            if abs(pos_start-debut) > 50000:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille} Attention !!! Diff position : {pos_start-debut}')
            else:
                print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille}')




            vecteur_Debut.append(debut)
            vecteur_Fin.append(fin)
            vecteur_longeur.append(taille)


print(len(vecteur_Debut))
result = pd.DataFrame()
result['Chromosome'] = list(range(1, 37))
result['start'] = vecteur_Debut 
result['end'] = vecteur_Fin
result['length'] = vecteur_longeur
#result.to_csv(f"../output/estimation/major/Major-Friedlin2021_X2X3.csv", index=False)
#result.to_csv(f"../output/estimation/major/Major-LV39c5_X2X3.csv", index=False)
#result.to_csv(f"../output/estimation/major/Major-SD751_X2X3.csv", index=False) 
#result.to_csv(f"../output/estimation/major/Major.csv", index=False)

1 : Commence 257127    Fini 258779  Taille : 1652
2 : Commence 265286    Fini 268347  Taille : 3061
3 : Commence 247037    Fini 253049  Taille : 6012
4 : Commence 125556    Fini 132214  Taille : 6658
5 : Commence 360714    Fini 367337  Taille : 6623
6 : Commence 122808    Fini 127353  Taille : 4545
7 : Commence 208630    Fini 214601  Taille : 5971
8 : Commence 487415    Fini 491520  Taille : 4105
9 : Commence 271664    Fini 277252  Taille : 5588
10 : Commence 296024    Fini 300542  Taille : 4518
11 : Commence 159688    Fini 161740  Taille : 2052
12 : Commence 287267    Fini 290888  Taille : 3621
13 : Commence 143539    Fini 145503  Taille : 1964
14 : Commence 158327    Fini 162610  Taille : 4283
15 : Commence 323047    Fini 329621  Taille : 6574
16 : Commence 340157    Fini 343824  Taille : 3667
17 : Commence 341042    Fini 344013  Taille : 2971
18 : Commence 443679    Fini 449382  Taille : 5703
19 : Commence 623659    Fini 630786  Taille : 7127
20 : Commence 523188    Fini 526438  Tai